# Genre classifier

This notebook selects a lyrics-based model and creates the test predictions.


## 1. Setup

In [1]:
import os
import sys
import re
import gc
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score, confusion_matrix

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 2026


## Split the data

A stratified 20% holdout is saved for the final check.


In [2]:
with open('../tmp/preprocessed_data.pickle', 'rb') as f:
    data = pickle.load(f)

X_text = np.asarray(data['train_clean_genre'], dtype=object)
X_test_text = np.asarray(data['test_clean_genre'], dtype=object)
y = np.asarray(data['y_train'])
classes = np.asarray(data['classes'])
test_ids = pd.read_csv('../dataset/test.csv', usecols=['id'])['id'].to_numpy()

train_idx, val_idx = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
y_tr, y_val = y[train_idx], y[val_idx]

select_train_pos, select_val_pos = train_test_split(
    np.arange(len(train_idx)), test_size=0.2, random_state=RANDOM_STATE + 1, stratify=y_tr
)
select_train_idx = train_idx[select_train_pos]
select_val_idx = train_idx[select_val_pos]
y_select_tr, y_select_val = y[select_train_idx], y[select_val_idx]

print(f"Classes: {classes.tolist()}")
print(f"Development fold: {len(train_idx):,} | Assessment fold: {len(val_idx):,} | Test: {len(X_test_text):,}")
print(f"Inner selection split: {len(select_train_idx):,} train | {len(select_val_idx):,} validation")


Classes: ['country', 'misc', 'pop', 'rap', 'rb', 'rock']
Development fold: 107,973 | Assessment fold: 26,994 | Test: 33,742
Inner selection split: 86,378 train | 21,595 validation


## Prepare model text

We remove transcription markers before comparing features and models.


In [ ]:
base_tfidf_params = dict(
    ngram_range=(1, 2), min_df=2, max_df=0.85, max_features=100_000,
    sublinear_tf=True, norm='l2', token_pattern=r"(?u)\b\w+\b"
)

def strip_transcription_markers(text):
    text = re.sub(r"\brepeat\s+chorus\b", " ", str(text), flags=re.IGNORECASE)
    text = re.sub(r"\b(?:chorus|verse)(?:\s+\d+)?\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bnbsp\b", " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

X_model_text = np.asarray([strip_transcription_markers(x) for x in X_text], dtype=object)
X_test_model_text = np.asarray([strip_transcription_markers(x) for x in X_test_text], dtype=object)

select_vec_raw = TfidfVectorizer(**base_tfidf_params)
X_select_raw_tr = select_vec_raw.fit_transform(X_text[select_train_idx])
X_select_raw_val = select_vec_raw.transform(X_text[select_val_idx])

select_vec_clean = TfidfVectorizer(**base_tfidf_params)
X_select_tr = select_vec_clean.fit_transform(X_model_text[select_train_idx])
X_select_val = select_vec_clean.transform(X_model_text[select_val_idx])

ablation_rows = []
for label, A_tr, A_val in [
    ('Original cleaned text', X_select_raw_tr, X_select_raw_val),
    ('Plain-text markers removed', X_select_tr, X_select_val),
]:
    probe = LinearSVC(C=0.1, class_weight='balanced').fit(A_tr, y_select_tr)
    ablation_rows.append({
        'Text version': label,
        'Selection Macro-F1': f1_score(y_select_val, probe.predict(A_val), average='macro')
    })

marker_ablation = pd.DataFrame(ablation_rows)
for _, row in marker_ablation.iterrows():
    print(f"{row['Text version']}: Macro-F1 {row['Selection Macro-F1']:.4f}")
print(f"Inner vocabulary after marker removal: {X_select_tr.shape[1]:,} features")


### Check chi-squared feature selection


In [ ]:
chi2_scores, _ = chi2(X_select_tr, y_select_tr)
feature_rank = np.argsort(-np.nan_to_num(chi2_scores))

k_values = sorted(set([5_000, 20_000, 50_000, X_select_tr.shape[1]]))
chi2_results = []
for k in k_values:
    cols = feature_rank[:k]
    clf = LinearSVC(C=0.1, class_weight='balanced').fit(X_select_tr[:, cols], y_select_tr)
    preds = clf.predict(X_select_val[:, cols])
    chi2_results.append({'k': k, 'Macro-F1': f1_score(y_select_val, preds, average='macro')})

chi2_df = pd.DataFrame(chi2_results)
ax = sns.lineplot(data=chi2_df, x='k', y='Macro-F1', marker='o')
ax.set_xscale('log')
ax.set_title('Inner selection Macro-F1 vs. number of Chi-squared selected features', weight='bold')
ax.set_xlabel('k (log scale)')
plt.tight_layout()
plt.show()


## Compare classifiers


In [ ]:
top100k = feature_rank[:min(100_000, len(feature_rank))]

models = {
    'ComplementNB (alpha=0.3)': (ComplementNB(alpha=0.3), None),
    'SGD modified_huber': (SGDClassifier(loss='modified_huber', alpha=2e-5, class_weight='balanced',
                                         random_state=RANDOM_STATE, max_iter=1000, tol=1e-3), None),
    'LinearSVC (C=0.1)': (LinearSVC(C=0.1, class_weight='balanced'), None),
    'LogisticRegression (C=10, top chi2 features)': (
        LogisticRegression(C=10, class_weight='balanced', max_iter=500), top100k
    ),
}

comparison = []
for name, (model, cols) in models.items():
    A_tr = X_select_tr if cols is None else X_select_tr[:, cols]
    A_val = X_select_val if cols is None else X_select_val[:, cols]
    model.fit(A_tr, y_select_tr)
    preds = model.predict(A_val)
    comparison.append({
        'Model': name,
        'Selection Macro-F1': f1_score(y_select_val, preds, average='macro'),
    })

comparison_df = pd.DataFrame(comparison).sort_values('Selection Macro-F1', ascending=False).reset_index(drop=True)
winner = comparison_df.iloc[0]
print(f"Best baseline: {winner['Model']} (Macro-F1 {winner['Selection Macro-F1']:.3f}).")

del X_select_raw_tr, X_select_raw_val, X_select_tr, X_select_val
del select_vec_raw, select_vec_clean


## Choose word and character features

The search compares word TF-IDF with character n-grams on the development data.


In [ ]:
word_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_df=0.85, sublinear_tf=True, norm='l2', token_pattern=r"(?u)\b\w+\b")),
    ('clf', LinearSVC(class_weight='balanced')),
])
word_param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.05, 0.1, 0.2],
}
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
word_grid = GridSearchCV(
    word_pipe, word_param_grid, scoring='f1_macro', cv=inner_cv,
    n_jobs=2, pre_dispatch='2*n_jobs', refit=True, verbose=1
)
word_grid.fit(X_model_text[train_idx], y_tr)

word_best_params = word_grid.best_params_.copy()
word_cv_score = float(word_grid.best_score_)
word_cv_std = float(word_grid.cv_results_['std_test_score'][word_grid.best_index_])
word_best_C = float(word_best_params['clf__C'])
word_tfidf_params = {
    'ngram_range': word_best_params['tfidf__ngram_range'],
    'min_df': int(word_best_params['tfidf__min_df']),
    'max_df': 0.85,
    'sublinear_tf': True,
    'norm': 'l2',
    'token_pattern': r"(?u)\b\w+\b",
}
print(f"Best word-only inner CV Macro-F1: {word_cv_score:.4f} ± {word_cv_std:.4f}")
del word_grid, word_pipe
_ = gc.collect()

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5), min_df=3, max_features=300_000,
    sublinear_tf=True, norm='l2'
)
char_pipe = Pipeline([
    ('features', FeatureUnion([
        ('word', TfidfVectorizer(**word_tfidf_params)),
        ('char', char_vectorizer),
    ])),
    ('clf', LinearSVC(class_weight='balanced')),
])
char_param_grid = {
    'features__transformer_weights': [
        {'word': 1.0, 'char': 0.25},
        {'word': 1.0, 'char': 0.5},
        {'word': 1.0, 'char': 1.0},
    ],
    'clf__C': [0.05, 0.1, 0.2],
}
char_grid = GridSearchCV(
    char_pipe, char_param_grid, scoring='f1_macro', cv=inner_cv,
    n_jobs=2, pre_dispatch='2*n_jobs', refit=True, verbose=1
)
char_grid.fit(X_model_text[train_idx], y_tr)

char_cv_score = float(char_grid.best_score_)
char_cv_std = float(char_grid.cv_results_['std_test_score'][char_grid.best_index_])
char_best_C = float(char_grid.best_params_['clf__C'])
char_weight = float(char_grid.best_params_['features__transformer_weights']['char'])
use_char_model = char_cv_score > word_cv_score

print(f"Word + character CV Macro-F1: {char_cv_score:.4f} ± {char_cv_std:.4f}")

if use_char_model:
    best_tfidf_params = word_tfidf_params.copy()
    best_C = char_best_C
    print('Selected word + character features based on higher inner CV Macro-F1.')
else:
    best_tfidf_params = word_tfidf_params.copy()
    best_C = word_best_C
    print('Selected word-only features; the character branch did not improve inner CV Macro-F1.')

char_params = {
    'analyzer': 'char_wb', 'ngram_range': (3, 5), 'min_df': 3,
    'max_features': 300_000, 'sublinear_tf': True, 'norm': 'l2'
}



## Test lyric-structure features

We compare lyric length, word variety, repeated lines, and line length against the selected text model. We keep them only if they improve validation scores consistently. We keep them only if they improve validation scores consistently.


In [ ]:
raw_train_lyrics = pd.read_csv('../dataset/train.csv', usecols=['lyrics'])['lyrics'].fillna('').astype(str).to_numpy()
raw_test_lyrics = pd.read_csv('../dataset/test.csv', usecols=['lyrics'])['lyrics'].fillna('').astype(str).to_numpy()
assert len(raw_train_lyrics) == len(X_model_text) == len(y)
assert len(raw_test_lyrics) == len(X_test_model_text)

X_model_pairs = np.column_stack([X_model_text, raw_train_lyrics])
X_test_pairs = np.column_stack([X_test_model_text, raw_test_lyrics])

STYLE_FEATURE_NAMES = [
    'log_word_count', 'unique_word_ratio', 'repeated_line_share',
    'median_words_per_line', 'short_line_share',
]
SECTION_HEADER = re.compile(
    r"^[\[\(\{]?\s*(?:pre[- ]?chorus|post[- ]?chorus|intro|verse|chorus|hook|bridge|outro|refrain|instrumental|spoken|interlude)(?:\s+\d+)?\s*:?\s*[\]\)\}]?$",
    flags=re.IGNORECASE,
)
WORD_PATTERN = re.compile(r"(?u)\b\w+\b")

class PairColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, column=0):
        self.column = column

    def fit(self, X, y=None):
        values = np.asarray(X, dtype=object)
        if values.ndim != 2 or self.column >= values.shape[1]:
            raise ValueError('Expected a 2-D array containing the requested paired input column.')
        self.n_features_in_ = values.shape[1]
        self.is_fitted_ = True
        return self

    def transform(self, X):
        values = np.asarray(X, dtype=object)
        return values[:, self.column]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(['cleaned_text' if self.column == 0 else 'raw_lyrics'], dtype=object)

class LyricStyleFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        rows = []
        for lyric in np.asarray(X, dtype=object).reshape(-1):
            text = '' if lyric is None else str(lyric)
            line_words = []
            for line in text.splitlines():
                stripped = line.strip()
                if not stripped or SECTION_HEADER.fullmatch(stripped):
                    continue
                words = WORD_PATTERN.findall(stripped.lower())
                if words:
                    line_words.append(words)

            words = [word for line in line_words for word in line]
            line_count = len(line_words)
            if words and line_count:
                normalized_lines = [' '.join(line) for line in line_words]
                counts = {}
                for line in normalized_lines:
                    counts[line] = counts.get(line, 0) + 1
                unique_ratio = len(set(words)) / len(words)
                repeated_share = sum(counts[line] > 1 for line in normalized_lines) / line_count
                median_line_length = float(np.median([len(line) for line in line_words]))
                short_line_share = sum(len(line) <= 4 for line in line_words) / line_count
            else:
                unique_ratio = repeated_share = median_line_length = short_line_share = 0.0

            rows.append([
                float(np.log1p(len(words))), unique_ratio, repeated_share,
                median_line_length, short_line_share,
            ])
        return pd.DataFrame(rows, columns=STYLE_FEATURE_NAMES)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(STYLE_FEATURE_NAMES, dtype=object)

class FeatureMultiplier(BaseEstimator, TransformerMixin):
    def __init__(self, weight=1.0):
        self.weight = weight

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float) * self.weight

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(STYLE_FEATURE_NAMES, dtype=object)
        return np.asarray(input_features, dtype=object)

def make_feature_transformer(style_weight=None):
    paired_input = style_weight is not None
    word_features = TfidfVectorizer(**best_tfidf_params)
    if paired_input:
        word_features = Pipeline([
            ('select_text', PairColumnSelector(column=0)),
            ('tfidf', word_features),
        ])

    if not use_char_model and not paired_input:
        return word_features

    branches = [('word', word_features)]
    weights = {'word': 1.0}
    if use_char_model:
        char_features = TfidfVectorizer(**char_params)
        if paired_input:
            char_features = Pipeline([
                ('select_text', PairColumnSelector(column=0)),
                ('tfidf', char_features),
            ])
        branches.append(('char', char_features))
        weights['char'] = char_weight

    if paired_input:
        branches.append(('style', Pipeline([
            ('select_lyrics', PairColumnSelector(column=1)),
            ('statistics', LyricStyleFeatures()),
            ('scale', StandardScaler()),
            ('weight', FeatureMultiplier(weight=style_weight)),
        ])))

    return FeatureUnion(branches, transformer_weights=weights)

style_cv = inner_cv
baseline_pipeline = Pipeline([
    ('features', make_feature_transformer()),
    ('clf', LinearSVC(C=best_C, class_weight='balanced', tol=1e-3, max_iter=2000)),
])
baseline_fold_scores = cross_val_score(
    baseline_pipeline, X_model_text[train_idx], y_tr,
    scoring='f1_macro', cv=style_cv, n_jobs=-1,
)
style_weights = [0.1, 0.25, 0.5]
style_fold_scores = {}
for style_weight in style_weights:
    style_pipeline = Pipeline([
        ('features', make_feature_transformer(style_weight=style_weight)),
        ('clf', LinearSVC(C=best_C, class_weight='balanced', tol=1e-3, max_iter=2000)),
    ])
    style_fold_scores[style_weight] = cross_val_score(
        style_pipeline, X_model_pairs[train_idx], y_tr,
        scoring='f1_macro', cv=style_cv, n_jobs=-1,
    )

best_style_weight = max(style_fold_scores, key=lambda weight: style_fold_scores[weight].mean())
best_style_scores = style_fold_scores[best_style_weight]
style_mean_gain = float(np.mean(best_style_scores - baseline_fold_scores))
style_fold_wins = int(np.sum(best_style_scores > baseline_fold_scores))
use_style_features = style_mean_gain >= 0.003 and style_fold_wins >= 2
selected_style_weight = best_style_weight if use_style_features else None
print(
    f"Best style-feature weight: {best_style_weight:g}. Macro-F1 changed by {style_mean_gain:+.4f} on average "
    f"and improved on {style_fold_wins}/3 folds. "
    f"{'Using the style features.' if use_style_features else 'Keeping the current model.'}"
)

X_classifier_inputs = X_model_pairs if use_style_features else X_model_text
X_test_classifier_inputs = X_test_pairs if use_style_features else X_test_model_text
vec = make_feature_transformer(style_weight=selected_style_weight)
t0 = time.time()
X_tr = vec.fit_transform(X_classifier_inputs[train_idx])
X_val = vec.transform(X_classifier_inputs[val_idx])
feature_description = 'word + character TF-IDF' if use_char_model else 'word TF-IDF'
if use_style_features:
    feature_description += f' + lyric style (weight={selected_style_weight:g})'
print(f"Selected {feature_description}: {X_tr.shape[1]:,} features; vectorised in {time.time()-t0:.0f}s")


## Adjust class scores

Small class offsets are tuned from out-of-fold development predictions.


In [8]:
offset_pipeline = Pipeline([
    ('features', make_feature_transformer(style_weight=selected_style_weight)),
    ('clf', LinearSVC(C=best_C, class_weight='balanced', tol=1e-3, max_iter=2000)),
])
oof_scores = cross_val_predict(
    offset_pipeline, X_classifier_inputs[train_idx], y_tr, method='decision_function',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + 2),
    n_jobs=-1
)

def tune_offsets(scores, y_true, grid=np.arange(-0.5, 0.51, 0.05), n_rounds=2):
    offsets = np.zeros(scores.shape[1])
    for _ in range(n_rounds):
        for c in range(scores.shape[1]):
            best_f1 = f1_score(y_true, (scores + offsets).argmax(1), average='macro')
            best_o = offsets[c]
            for o in grid:
                trial = offsets.copy()
                trial[c] = o
                f1 = f1_score(y_true, (scores + trial).argmax(1), average='macro')
                if f1 > best_f1:
                    best_f1, best_o = f1, o
            offsets[c] = best_o
    return offsets

offsets = tune_offsets(oof_scores, y_tr)
print("Tuned offsets:", {str(c): round(float(o), 2) for c, o in zip(classes, offsets)})

svc = LinearSVC(C=best_C, class_weight='balanced', tol=1e-3, max_iter=2000).fit(X_tr, y_tr)
val_scores = svc.decision_function(X_val)
pred_plain = val_scores.argmax(1)
pred_offset = (val_scores + offsets).argmax(1)

print(f"\nOuter assessment Macro-F1 without offsets: {f1_score(y_val, pred_plain, average='macro'):.4f}")
print(f"Outer assessment Macro-F1 with offsets:    {f1_score(y_val, pred_offset, average='macro'):.4f}")


Tuned offsets: {'country': -0.05, 'misc': -0.25, 'pop': 0.0, 'rap': -0.15, 'rb': 0.0, 'rock': 0.05}

Outer assessment Macro-F1 without offsets: 0.5351
Outer assessment Macro-F1 with offsets:    0.5446


## Check performance on the holdout


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1.3, 1]})

cm = confusion_matrix(y_val, pred_offset, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[0], cbar=False)
axes[0].set_title('Confusion matrix (row-normalised = recall)', weight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

per_class = pd.DataFrame({
    'Genre': classes,
    'F1': f1_score(y_val, pred_offset, average=None),
    'Share of training data': np.bincount(y_tr) / len(y_tr),
}).sort_values('F1', ascending=False)
sns.barplot(data=per_class, x='F1', y='Genre', ax=axes[1], color='steelblue')
for i, row in enumerate(per_class.itertuples()):
    axes[1].text(row.F1 + 0.01, i, f"{row.F1:.2f}  (train share {row._3:.1%})", va='center', fontsize=9)
axes[1].set_xlim(0, 1)
axes[1].set_title('F1 per genre (training share in brackets)', weight='bold')

plt.tight_layout()
plt.show()


## Inspect `misc` examples


In [11]:
raw_train = pd.read_csv('../dataset/train.csv', usecols=['title', 'artist', 'lyrics', 'tag'])
misc_label = list(classes).index('misc')

val_view = raw_train.iloc[val_idx].assign(pred=classes[pred_offset])
misc_val = val_view[val_view['tag'] == 'misc']
print("Where true `misc` songs end up:")
print(misc_val['pred'].value_counts(normalize=True).round(3).to_string())

print("\nSample of `misc` songs:")
for row in misc_val.sample(6, random_state=RANDOM_STATE).itertuples():
    snippet = ' '.join(str(row.lyrics).split())[:140]
    print(f"- [{row.pred:>7}] {row.title} - {row.artist}: {snippet}...")

Where true `misc` songs end up:
pred
misc       0.558
pop        0.215
rock       0.119
rap        0.080
country    0.016
rb         0.013

Sample of `misc` songs:
- [   misc] Özils Career Honours - Mesut zil: Arsenal FCWinner: FA Cup (3): 2013–14, 2014-15, 2016-17 FA Community Shield: 2014, 2015Real Madrid CFLa Liga: 2011–12 Copa del Rey: 2010–11 ...
- [   misc] The House Behind the Cedars Chap. 6 - Charles Chesnutt: The Queen Of Love And Beauty Shortly after luncheon, Rena had a visitor in the person of Mrs. Newberry, a vivacious young widow of the town,...
- [   misc] Barchester Towers Chap. 53 - Anthony Trollope: Conclusion The end of a novel, like the end of a children's dinner party, must be made up of sweetmeats and sugar-plums. There is now nothin...
- [   misc] Genius Dream Album - ScHoolboy Q - Genius Dream Album: Standard Edition Deluxe Edition Executive Producers: AntHony "Top Dawg" TiffitH, ScHoolboy Q All songs mixed & mastered by Derek "MixedByAli...
- [    pop] Negro Se

## Build the test submission


In [ ]:
final_vec = make_feature_transformer(style_weight=selected_style_weight)
X_full = final_vec.fit_transform(X_classifier_inputs)
X_test = final_vec.transform(X_test_classifier_inputs)

final_svc = LinearSVC(C=best_C, class_weight='balanced', tol=1e-3, max_iter=2000).fit(X_full, y)
test_pred = classes[(final_svc.decision_function(X_test) + offsets).argmax(1)]

submission = pd.DataFrame({'id': test_ids, 'tag': test_pred})
sample = pd.read_csv('../dataset/sample_submission.csv')
submission = sample[['id']].merge(submission, on='id', how='left')
assert len(submission) == len(sample) and submission['tag'].notna().all()

os.makedirs('../submissions', exist_ok=True)
submission.to_csv('../submissions/submission.csv', index=False)
print(f"Saved {len(submission):,} predictions to ../submissions/submission.csv")
